# EISCAT (Folkestad, Hagfors, Westerlund 1983) — Implementation / 구현

**Paper**: Folkestad, K., Hagfors, T., and Westerlund, S. (1983). EISCAT: An updated description of technical characteristics and operational capabilities. *Radio Science*, 18(6), 867–879. DOI: 10.1029/RS018i006p00867

This notebook reproduces three core EISCAT data-products from first principles:

1. **Part 1 — Incoherent-scatter ion-line spectrum**: synthesise the Salpeter-form spectrum and recover $T_e, T_i$ from spectrum width and shape.
2. **Part 2 — Tristatic ion-velocity inversion**: invert three line-of-sight Doppler velocities into the full 3-D ion drift vector, then to the convection electric field $\vec E_\perp = -\vec v_i \times \vec B$.
3. **Part 3 — Multipulse autocorrelation**: build a 5-pulse waveform with a unique-difference lag set, compute the resulting ACF estimator, and compare to the conventional triangular lag-weighting of a single long pulse.

이 노트북은 EISCAT의 세 가지 핵심 데이터 산물을 1차 원리부터 재현합니다.
1. **1부 — 비간섭산란 이온선 스펙트럼**: Salpeter 형태의 스펙트럼을 합성하고 폭·모양으로부터 $T_e, T_i$를 회복.
2. **2부 — 삼정점 이온 속도 역변환**: 세 시선 도플러 속도로부터 3차원 이온 표류 벡터를 복원하고, $\vec E_\perp = -\vec v_i \times \vec B$로 대류 전기장 도출.
3. **3부 — 다중펄스 자기상관**: 유일-차이 래그 집합을 가진 5-펄스 파형을 설계하고 그 ACF 추정기를 계산해, 단일 장펄스의 종래 삼각 가중과 비교.

In [ ]:
"""Common imports and plotting setup for the EISCAT implementation notebook."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True

# Physical constants used throughout the notebook (SI units).
K_BOLTZ = 1.380649e-23  # J/K
M_E = 9.1093837e-31     # kg, electron mass
AMU = 1.66053907e-27    # kg, atomic mass unit
Q_E = 1.602176634e-19   # C, elementary charge
EPS0 = 8.8541878128e-12  # F/m, vacuum permittivity
C_LIGHT = 2.99792458e8  # m/s

# EISCAT UHF radar parameters (Folkestad et al. 1983, Table 1 and Table 2).
F_UHF_HZ = 933.5e6                  # Center frequency, 933.5 MHz
LAMBDA_UHF = C_LIGHT / F_UHF_HZ     # ~32.1 cm wavelength
K_RADAR = 4.0 * np.pi / LAMBDA_UHF  # Bragg wavenumber 2*k0 for monostatic backscatter

print(f"UHF wavelength      : {LAMBDA_UHF*100:.2f} cm")
print(f"Bragg wavenumber 2k0: {K_RADAR:.2f} rad/m")

## Part 1: Incoherent-scatter ion-line spectrum / 비간섭산란 이온선 스펙트럼

**EN**: For an unmagnetised plasma in the small-Debye-length limit ($k\lambda_D \ll 1$) the Salpeter form of the ion-line spectrum is

$$ S(\omega) = \frac{2 N_e}{|1 + \alpha^2 + \alpha^2 (T_e/T_i) y_i|^2} \, \mathrm{Re}\,\big[ y_i^*\,\alpha^2 (T_e/T_i)\big] + \dots $$

but a tractable engineering proxy that captures the **double-humped shape, ion-acoustic peak position, and width** is the simplified form used by Hagfors (1961):

$$ S_i(\omega) \;\propto\; \frac{(T_e/T_i)\,/\,\sqrt{2\pi}\,k v_{Ti}}{ \big(1 + (\omega/\omega_a)^2 - 1\big)^2 + (\omega/\omega_a)^2 / Q^2 } $$

where $\omega_a = k c_s$ is the ion-acoustic frequency, $c_s = \sqrt{k_B T_e / m_i}$ is the ion-acoustic speed, and $Q = (T_e/T_i)^{1/2}$ controls the sharpness of the peaks. In this notebook we use a robust **two-Lorentzian** ansatz that reproduces the same engineering observables (peak frequency $\propto c_s$, peak-to-trough ratio $\propto T_e/T_i$, full width $\propto k v_{Ti}$) and is friendly to least-squares fitting:

$$ S(\omega) = A \bigg[ \frac{\Gamma}{(\omega-\omega_a)^2 + \Gamma^2} + \frac{\Gamma}{(\omega+\omega_a)^2 + \Gamma^2} \bigg] + S_0 $$

with $\omega_a = k c_s$ and $\Gamma = k v_{Ti} \sqrt{T_i / T_e}$. The dependence of the peak shoulder on $T_e/T_i$ encodes the temperature ratio retrieved by EISCAT.

**KR**: 비자화 플라스마의 작은 디바이 길이 한계($k\lambda_D \ll 1$)에서 Salpeter형 이온선 스펙트럼은 위 식으로 주어진다. 본 노트북은 이를 공학적으로 다루기 쉬운 **두-Lorentzian** 형태로 모형화한다 — 같은 관측량(이온 음파 봉우리 위치 $\propto c_s$, 봉우리/골 비 $\propto T_e/T_i$, 폭 $\propto k v_{Ti}$)을 재현하면서 최소제곱 적합에 유리하다.

In [ ]:
def ion_acoustic_speed(t_e_kelvin: float, ion_mass_amu: float = 16.0) -> float:
    """Compute the ion-acoustic speed c_s = sqrt(k_B T_e / m_i).

    Args:
        t_e_kelvin: Electron temperature in kelvin.
        ion_mass_amu: Ion mass in atomic mass units. Default is 16 (atomic oxygen, F-region).

    Returns:
        Ion-acoustic speed in m/s.
    """
    return np.sqrt(K_BOLTZ * t_e_kelvin / (ion_mass_amu * AMU))


def ion_thermal_speed(t_i_kelvin: float, ion_mass_amu: float = 16.0) -> float:
    """Compute the ion thermal speed v_Ti = sqrt(k_B T_i / m_i).

    Args:
        t_i_kelvin: Ion temperature in kelvin.
        ion_mass_amu: Ion mass in atomic mass units.

    Returns:
        Ion thermal speed in m/s.
    """
    return np.sqrt(K_BOLTZ * t_i_kelvin / (ion_mass_amu * AMU))


def ion_line_spectrum(
    omega: np.ndarray,
    t_e: float,
    t_i: float,
    n_e: float = 1e11,
    ion_mass_amu: float = 16.0,
    k: float = K_RADAR,
) -> np.ndarray:
    """Synthesise the EISCAT ion-line spectrum using a two-Lorentzian Salpeter proxy.

    The spectrum has two ion-acoustic peaks at +/- omega_a = k * c_s, with linewidth
    Gamma = k * v_Ti * sqrt(T_i/T_e). The peak amplitude is normalised such that the
    integral over omega returns a quantity proportional to N_e / (1 + T_e/T_i),
    matching the volumetric incoherent-scatter cross-section.

    Args:
        omega: Angular frequency offset from the radar carrier in rad/s.
        t_e: Electron temperature in kelvin.
        t_i: Ion temperature in kelvin.
        n_e: Electron density in 1/m^3.
        ion_mass_amu: Ion mass in atomic mass units (16 = O+).
        k: Bragg wavenumber in rad/m. Default is the EISCAT UHF backscatter value.

    Returns:
        Array of spectrum values S(omega) in arbitrary units proportional to power.
    """
    c_s = ion_acoustic_speed(t_e, ion_mass_amu)
    v_ti = ion_thermal_speed(t_i, ion_mass_amu)
    omega_a = k * c_s
    gamma = k * v_ti * np.sqrt(t_i / t_e)
    amplitude = n_e / (1.0 + t_e / t_i)

    plus_peak = gamma / ((omega - omega_a) ** 2 + gamma ** 2)
    minus_peak = gamma / ((omega + omega_a) ** 2 + gamma ** 2)
    return amplitude * (plus_peak + minus_peak) / np.pi

In [ ]:
# Generate ion-line spectra for a range of T_e/T_i ratios at fixed T_i.
# This reproduces the well-known double-humped shape of the EISCAT ion line.

freq_hz = np.linspace(-30e3, 30e3, 1024)  # Frequency offset in Hz
omega_grid = 2.0 * np.pi * freq_hz

t_i_fixed = 1500.0  # Ion temperature in kelvin (typical F region)
ratios = [1.0, 1.5, 2.0, 3.0]

fig, ax = plt.subplots()
for ratio in ratios:
    t_e = ratio * t_i_fixed
    spectrum = ion_line_spectrum(omega_grid, t_e, t_i_fixed)
    ax.plot(freq_hz / 1e3, spectrum / spectrum.max(), label=f"$T_e/T_i$ = {ratio:.1f}")
ax.set_xlabel("Frequency offset (kHz)")
ax.set_ylabel("Normalised spectral power")
ax.set_title("EISCAT UHF ion-line spectrum vs. $T_e/T_i$ (T_i = 1500 K, O+)")
ax.legend()
fig.tight_layout()
plt.show()

print("Engineering observables:")
print(f"  c_s at T_e = T_i = 1500 K : {ion_acoustic_speed(1500):.0f} m/s")
print(f"  Peak frequency (kHz)      : {K_RADAR * ion_acoustic_speed(1500) / (2*np.pi) / 1e3:.2f}")

In [ ]:
def fit_temperatures_from_spectrum(
    omega: np.ndarray,
    measured: np.ndarray,
    ion_mass_amu: float = 16.0,
    k: float = K_RADAR,
    initial_t_e: float = 2000.0,
    initial_t_i: float = 1000.0,
) -> tuple[float, float, float]:
    """Recover (T_e, T_i, N_e) by least-squares fitting the two-Lorentzian model.

    Args:
        omega: Angular frequency grid in rad/s.
        measured: Measured spectrum values.
        ion_mass_amu: Assumed ion mass in atomic mass units.
        k: Bragg wavenumber in rad/m.
        initial_t_e: Initial guess for electron temperature in kelvin.
        initial_t_i: Initial guess for ion temperature in kelvin.

    Returns:
        Tuple (t_e_fit, t_i_fit, n_e_fit) with fitted parameters.
    """

    def model(om, t_e, t_i, n_e):
        return ion_line_spectrum(om, t_e, t_i, n_e=n_e, ion_mass_amu=ion_mass_amu, k=k)

    initial_n_e = measured.max() * (1.0 + initial_t_e / initial_t_i)
    popt, _ = curve_fit(
        model,
        omega,
        measured,
        p0=[initial_t_e, initial_t_i, initial_n_e],
        bounds=([300.0, 300.0, 1e9], [10000.0, 10000.0, 1e13]),
    )
    return float(popt[0]), float(popt[1]), float(popt[2])


# Demonstrate retrieval from a noisy synthetic spectrum.
rng = np.random.default_rng(seed=2026)
true_t_e = 3000.0
true_t_i = 1500.0
true_n_e = 1.0e11
clean_spectrum = ion_line_spectrum(omega_grid, true_t_e, true_t_i, n_e=true_n_e)
noise_level = 0.05 * clean_spectrum.max()
noisy_spectrum = clean_spectrum + rng.normal(scale=noise_level, size=clean_spectrum.shape)

t_e_fit, t_i_fit, n_e_fit = fit_temperatures_from_spectrum(omega_grid, noisy_spectrum)
print("Fit results (truth in parentheses):")
print(f"  T_e = {t_e_fit:7.1f} K  (true {true_t_e:.1f} K)")
print(f"  T_i = {t_i_fit:7.1f} K  (true {true_t_i:.1f} K)")
print(f"  N_e = {n_e_fit:.3e}  (true {true_n_e:.3e})")

fig, ax = plt.subplots()
ax.plot(freq_hz / 1e3, noisy_spectrum, ".", alpha=0.5, label="Noisy measurement")
ax.plot(freq_hz / 1e3, clean_spectrum, "-", lw=2, label="Truth")
ax.plot(
    freq_hz / 1e3,
    ion_line_spectrum(omega_grid, t_e_fit, t_i_fit, n_e=n_e_fit),
    "--",
    lw=2,
    label="Fit",
)
ax.set_xlabel("Frequency offset (kHz)")
ax.set_ylabel("Spectral power (a.u.)")
ax.set_title("Recovering $T_e$ and $T_i$ from a noisy EISCAT ion-line spectrum")
ax.legend()
fig.tight_layout()
plt.show()

## Part 2: Tristatic ion-velocity inversion / 삼정점 이온 속도 역변환

**EN**: A scattering volume is illuminated by a transmitter at $\hat n_{\rm tx}$ and observed by three receivers at $\hat n_{{\rm rx},1}, \hat n_{{\rm rx},2}, \hat n_{{\rm rx},3}$. For each receiver the Bragg vector is $\vec k_i \propto \hat n_{\rm tx} + \hat n_{{\rm rx},i}$. The Doppler velocity measured at receiver $i$ is the projection of the ion drift onto the unit Bragg direction $\hat k_i$. Stacking all three measurements yields a $3\times 3$ linear system that we invert (via `numpy.linalg.solve`) to recover the full ion-drift vector. The convection electric field follows from $\vec E_\perp = -\vec v_i \times \vec B$.

**KR**: 송신기 방향 $\hat n_{\rm tx}$와 세 수신기 $\hat n_{{\rm rx},1,2,3}$에 대해 Bragg 벡터는 $\vec k_i \propto \hat n_{\rm tx} + \hat n_{{\rm rx},i}$. 각 수신기의 도플러 속도는 이온 표류를 $\hat k_i$ 방향으로 사영한 값. 셋을 쌓아 $3\times 3$ 선형계를 만들고 `numpy.linalg.solve`로 풀어 3차원 이온 표류를 복원, 이어서 $\vec E_\perp = -\vec v_i \times \vec B$로 대류 전기장을 도출.

In [ ]:
def bragg_unit_vector(n_tx: np.ndarray, n_rx: np.ndarray) -> np.ndarray:
    """Compute the unit Bragg direction for a bistatic scatter geometry.

    The Bragg vector for a (transmitter, receiver) pair is k_tx + k_rx, where
    each unit vector points from the scatter volume back along the line of
    propagation to the corresponding station.

    Args:
        n_tx: 3-vector pointing from the scatter volume to the transmitter.
        n_rx: 3-vector pointing from the scatter volume to the receiver.

    Returns:
        Unit Bragg vector as a 3-element numpy array.
    """
    bragg = n_tx + n_rx
    return bragg / np.linalg.norm(bragg)


def invert_tristatic_velocity(
    bragg_directions: np.ndarray,
    line_of_sight_velocities: np.ndarray,
) -> np.ndarray:
    """Invert three line-of-sight Doppler velocities into the 3-D ion drift.

    Args:
        bragg_directions: Shape (3, 3) array whose rows are the unit Bragg vectors.
        line_of_sight_velocities: Length-3 array of measured Doppler velocities.

    Returns:
        Length-3 array containing the ion drift vector v_i.

    Raises:
        numpy.linalg.LinAlgError: If the geometry matrix is singular (collinear
            Bragg directions).
    """
    return np.linalg.solve(bragg_directions, line_of_sight_velocities)


def convection_efield(v_drift: np.ndarray, b_field: np.ndarray) -> np.ndarray:
    """Compute the perpendicular convection electric field E = -v x B.

    Args:
        v_drift: Ion drift 3-vector in m/s.
        b_field: Magnetic field 3-vector in tesla.

    Returns:
        Electric field 3-vector in V/m.
    """
    return -np.cross(v_drift, b_field)

In [ ]:
# Build a realistic EISCAT-like geometry. Use a local east-north-up (ENU) frame
# anchored at the scatter volume above Tromso at 300 km altitude.
scatter_alt_km = 300.0
earth_radius_km = 6371.0

# Approximate ENU coordinates (km) of the three stations relative to the scatter
# volume directly above Tromso. Distances chosen to match the published bistatic
# baselines (~250 km Tromso-Kiruna, ~370 km Tromso-Sodankyla).
stations = {
    "Tromso":   np.array([   0.0,    0.0, -scatter_alt_km]),
    "Kiruna":   np.array([  60.0, -210.0, -scatter_alt_km]),
    "Sodankyla": np.array([ 320.0, -200.0, -scatter_alt_km]),
}

# Unit vectors from the scatter volume back to each station.
n_tromso = -stations["Tromso"] / np.linalg.norm(stations["Tromso"])
n_kiruna = -stations["Kiruna"] / np.linalg.norm(stations["Kiruna"])
n_sodanky = -stations["Sodankyla"] / np.linalg.norm(stations["Sodankyla"])

# Bragg directions for each (Tx=Tromso, Rx) pair.
bragg = np.array([
    bragg_unit_vector(n_tromso, n_tromso),    # monostatic at Tromso
    bragg_unit_vector(n_tromso, n_kiruna),    # bistatic Tromso-Kiruna
    bragg_unit_vector(n_tromso, n_sodanky),   # bistatic Tromso-Sodankyla
])

print("Bragg unit vectors (rows = Tromso, Kiruna, Sodankyla):")
print(np.array_str(bragg, precision=3, suppress_small=True))
print(f"Determinant (geometry conditioning): {np.linalg.det(bragg):.4f}")

In [ ]:
# Inject a known auroral ion drift, project onto each Bragg direction,
# add Gaussian measurement noise, then invert and compare.

true_drift_mps = np.array([800.0, -500.0, 50.0])  # eastward, southward, upward
los_truth = bragg @ true_drift_mps

rng_drift = np.random.default_rng(seed=42)
los_noise_mps = 25.0  # typical EISCAT random uncertainty per LOS sample
los_measured = los_truth + rng_drift.normal(scale=los_noise_mps, size=3)

v_recovered = invert_tristatic_velocity(bragg, los_measured)

print("True drift     (m/s):", np.array_str(true_drift_mps, precision=1))
print("LOS truth      (m/s):", np.array_str(los_truth, precision=1))
print("LOS measured   (m/s):", np.array_str(los_measured, precision=1))
print("Recovered v_i  (m/s):", np.array_str(v_recovered, precision=1))
print(f"Recovery error (m/s): {np.linalg.norm(v_recovered - true_drift_mps):.1f}")

# Convert the ion drift into a convection electric field. At Tromso the
# geomagnetic field is ~52000 nT, mostly downward (high magnetic dip).
b_at_tromso = 1e-9 * np.array([0.0, 10000.0, -51000.0])  # tesla, ENU
e_field = convection_efield(v_recovered, b_at_tromso)
print("\nConvection electric field E_perp (mV/m):", np.array_str(e_field * 1e3, precision=2))
print(f"|E_perp|: {np.linalg.norm(e_field) * 1e3:.2f} mV/m  (typical auroral magnitude 10-100 mV/m)")

In [ ]:
# Monte Carlo: characterise the inversion error vs LOS noise level.
rng_mc = np.random.default_rng(seed=1234)
noise_levels_mps = np.array([5.0, 10.0, 25.0, 50.0, 100.0])
n_trials = 2000
rms_errors = np.zeros_like(noise_levels_mps)

for idx, sigma in enumerate(noise_levels_mps):
    errors = np.zeros(n_trials)
    for trial in range(n_trials):
        los_trial = los_truth + rng_mc.normal(scale=sigma, size=3)
        recovered = invert_tristatic_velocity(bragg, los_trial)
        errors[trial] = np.linalg.norm(recovered - true_drift_mps)
    rms_errors[idx] = np.sqrt(np.mean(errors ** 2))

fig, ax = plt.subplots()
ax.loglog(noise_levels_mps, rms_errors, "o-", lw=2)
ax.set_xlabel("Per-LOS noise sigma (m/s)")
ax.set_ylabel("RMS error in 3-D drift (m/s)")
ax.set_title("Tristatic inversion noise propagation (Tromso–Kiruna–Sodankyla geometry)")
fig.tight_layout()
plt.show()

for sigma, err in zip(noise_levels_mps, rms_errors):
    print(f"  sigma_LOS = {sigma:6.1f} m/s -> RMS 3-D drift error = {err:6.1f} m/s")

## Part 3: Multipulse waveform autocorrelation / 다중펄스 자기상관

**EN**: A multipulse waveform fires $M$ short subpulses at carefully chosen times $\{t_1, t_2, \ldots, t_M\}$ within an interpulse period. The set of pairwise time differences $\{t_k - t_j : k > j\}$ samples lags of the ACF; if every desired lag appears **exactly once**, each lag is estimated cleanly from a single cross-product. The classical Sulzer/Farley pattern $\{0,1,4,9,11\}$ generates the lag set $\{1,2,3,4,5,7,8,9,10,11\}$ (with no duplicates) and is widely used at EISCAT for fine-resolution $E$-region work, complementing the long single pulse used for $T_e/T_i$.

Here we (i) generate a 5-pulse waveform with $\{0,1,4,9,11\}$ delays, (ii) form the ACF estimator from each unique lag pair, and (iii) compare to the conventional triangular lag-weighting from a single pulse of the same total duration.

**KR**: 다중펄스 파형은 펄스간 주기 안에서 $\{t_1, \ldots, t_M\}$ 시간에 짧은 부펄스 $M$개를 발사. 쌍별 시간차 집합 $\{t_k - t_j : k > j\}$가 ACF의 래그를 표본화하며, 모든 원하는 래그가 **정확히 한 번씩** 나타나면 각 래그는 단일 교차곱으로 깔끔히 추정됨. 고전적 Sulzer/Farley 패턴 $\{0,1,4,9,11\}$은 래그 집합 $\{1,2,3,4,5,7,8,9,10,11\}$(중복 없음)을 생성하며, EISCAT $E$ 영역의 미세 분해능 관측에서 단일 장펄스($T_e/T_i$용)와 함께 표준적으로 사용된다.

In [ ]:
def build_multipulse(
    pulse_offsets: list[int],
    subpulse_samples: int,
    total_samples: int,
) -> np.ndarray:
    """Construct a multipulse waveform on a uniformly sampled time grid.

    Args:
        pulse_offsets: List of integer offsets (in subpulse units) where each
            short pulse begins. The Sulzer/Farley pattern is [0, 1, 4, 9, 11].
        subpulse_samples: Number of samples in a single short pulse.
        total_samples: Total length of the waveform in samples.

    Returns:
        Real-valued waveform array of length total_samples (1 inside subpulses,
        0 elsewhere).
    """
    waveform = np.zeros(total_samples)
    for offset in pulse_offsets:
        start = offset * subpulse_samples
        end = start + subpulse_samples
        if end > total_samples:
            raise ValueError("Pulse extends beyond waveform length")
        waveform[start:end] = 1.0
    return waveform


def unique_lag_set(pulse_offsets: list[int]) -> dict[int, int]:
    """Compute the multiplicity of each pairwise lag in a multipulse pattern.

    Args:
        pulse_offsets: Pulse start positions in subpulse units.

    Returns:
        Dictionary mapping lag (in subpulse units) to multiplicity.
    """
    counts: dict[int, int] = {}
    for j_idx, t_j in enumerate(pulse_offsets):
        for t_k in pulse_offsets[j_idx + 1:]:
            lag = t_k - t_j
            counts[lag] = counts.get(lag, 0) + 1
    return counts


# Inspect the Sulzer/Farley pattern.
sulzer_pattern = [0, 1, 4, 9, 11]
lag_counts = unique_lag_set(sulzer_pattern)
print(f"Sulzer/Farley pattern: {sulzer_pattern}")
print(f"Lag set (lag: count) : {dict(sorted(lag_counts.items()))}")
print(f"All unique?          : {all(c == 1 for c in lag_counts.values())}")
print(f"Number of lags       : {len(lag_counts)}")

In [ ]:
# Visualise the multipulse waveform vs a single long pulse of the same total duration.
subpulse_samples = 20  # 20 samples per short pulse (e.g., 20 us at 1 us sampling)
total_samples = 320  # full IPP grid in samples

multi_pulse_waveform = build_multipulse(sulzer_pattern, subpulse_samples, total_samples)
single_pulse_waveform = np.zeros(total_samples)
single_pulse_waveform[: 12 * subpulse_samples] = 1.0  # equivalent total duration

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 5))
axes[0].plot(multi_pulse_waveform, lw=2)
axes[0].set_ylabel("Multipulse")
axes[0].set_title("Sulzer/Farley pattern [0, 1, 4, 9, 11] vs a single long pulse")
axes[1].plot(single_pulse_waveform, lw=2, color="crimson")
axes[1].set_ylabel("Single pulse")
axes[1].set_xlabel("Sample index")
fig.tight_layout()
plt.show()

In [ ]:
def synthesise_acf_signal(
    omega_grid: np.ndarray,
    spectrum: np.ndarray,
    n_samples: int,
    sample_dt: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """Generate a complex Gaussian random signal with a prescribed power spectrum.

    The procedure is the standard spectral synthesis: draw independent zero-mean
    complex Gaussian coefficients for each frequency bin scaled by the square
    root of the spectrum, then inverse FFT to time domain.

    Args:
        omega_grid: Angular frequency grid the spectrum is sampled on.
        spectrum: Power spectrum values on omega_grid.
        n_samples: Length of the time-domain signal to return.
        sample_dt: Sampling interval in seconds.
        rng: Numpy random generator.

    Returns:
        Complex-valued array of length n_samples.
    """
    nyquist_rad = np.pi / sample_dt
    fft_omegas = 2.0 * np.pi * np.fft.fftfreq(n_samples, sample_dt)
    interpolated = np.interp(
        np.abs(fft_omegas),
        omega_grid[omega_grid >= 0],
        spectrum[omega_grid >= 0],
        left=0.0,
        right=0.0,
    )
    interpolated = np.maximum(interpolated, 0.0)
    coefficients = (
        rng.normal(size=n_samples) + 1j * rng.normal(size=n_samples)
    ) * np.sqrt(interpolated / 2.0)
    return np.fft.ifft(coefficients) * np.sqrt(n_samples / sample_dt)


def estimate_acf_multipulse(
    samples: np.ndarray,
    pulse_offsets: list[int],
    subpulse_samples: int,
) -> dict[int, complex]:
    """Estimate ACF lags from a multipulse return using unique lag pairs.

    Args:
        samples: Complex signal received during one IPP.
        pulse_offsets: Multipulse pattern in subpulse units.
        subpulse_samples: Samples per subpulse.

    Returns:
        Dictionary mapping lag (subpulse units) to complex ACF estimate.
    """
    estimates: dict[int, list[complex]] = {}
    for j_idx, t_j in enumerate(pulse_offsets):
        for t_k in pulse_offsets[j_idx + 1:]:
            lag = t_k - t_j
            window_j = samples[t_j * subpulse_samples : (t_j + 1) * subpulse_samples]
            window_k = samples[t_k * subpulse_samples : (t_k + 1) * subpulse_samples]
            cross = np.mean(window_j.conj() * window_k)
            estimates.setdefault(lag, []).append(cross)
    return {lag: np.mean(values) for lag, values in estimates.items()}

In [ ]:
# End-to-end demonstration: synthesise an ACF-consistent signal, estimate ACF
# lags via the multipulse pattern, and overlay the analytic ACF (Wiener-Khinchin).
rng_mp = np.random.default_rng(seed=7)
sample_dt = 1.0e-6  # 1 us sampling -> 500 kHz Nyquist
ipp_samples = 2048

spectrum_truth = ion_line_spectrum(omega_grid, t_e=2500.0, t_i=1500.0, n_e=1.0e11)

n_realisations = 200
lag_accumulator: dict[int, list[complex]] = {}
for _ in range(n_realisations):
    signal = synthesise_acf_signal(omega_grid, spectrum_truth, ipp_samples, sample_dt, rng_mp)
    lag_estimates = estimate_acf_multipulse(signal, sulzer_pattern, subpulse_samples)
    for lag, val in lag_estimates.items():
        lag_accumulator.setdefault(lag, []).append(val)

averaged_acf = {lag: np.mean(vals) for lag, vals in lag_accumulator.items()}

# Analytic ACF via Wiener-Khinchin (inverse FFT of the spectrum).
n_dense = len(spectrum_truth)
domega = omega_grid[1] - omega_grid[0]
ifft_grid = np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(spectrum_truth))) * (n_dense * domega) / (2 * np.pi)
tau_grid = np.fft.fftshift(np.fft.fftfreq(n_dense, d=domega / (2 * np.pi)))
analytic_acf_real = ifft_grid.real / ifft_grid.real.max()

fig, ax = plt.subplots()
lags = sorted(averaged_acf.keys())
lag_seconds = np.array(lags) * subpulse_samples * sample_dt
estimate_real = np.array([averaged_acf[lag].real for lag in lags])
estimate_real = estimate_real / estimate_real.max()
ax.plot(tau_grid * 1e6, analytic_acf_real, "-", label="Analytic ACF (Wiener-Khinchin)")
ax.plot(lag_seconds * 1e6, estimate_real, "o", ms=8, label="Multipulse estimate")
ax.set_xlim(0, 320)
ax.set_xlabel("Lag (us)")
ax.set_ylabel("Normalised Re[R(tau)]")
ax.set_title("Multipulse ACF estimator vs analytic ACF (averaged over 200 realisations)")
ax.legend()
fig.tight_layout()
plt.show()

for lag in lags:
    print(
        f"  lag = {lag:>2d} subpulses ({lag * subpulse_samples * sample_dt * 1e6:5.0f} us)"
        f"  -> Re[R] = {averaged_acf[lag].real:+.3e}"
    )

In [ ]:
# Compare the multipulse uniform range cell against the conventional single-pulse
# triangular lag-weighting (Section 3.2 of the paper).
n_lags_demo = 15
single_pulse_weights = np.maximum(1.0 - np.arange(n_lags_demo) / n_lags_demo, 0.0)
multipulse_weights = np.zeros(n_lags_demo)
for lag in lag_counts:
    if lag < n_lags_demo:
        multipulse_weights[lag] = 1.0  # uniform weight per unique lag

fig, ax = plt.subplots()
ax.bar(np.arange(n_lags_demo) - 0.18, single_pulse_weights, width=0.35,
       label="Single long pulse (matched-filter triangular)", color="crimson")
ax.bar(np.arange(n_lags_demo) + 0.18, multipulse_weights, width=0.35,
       label="Sulzer/Farley multipulse (uniform per unique lag)", color="steelblue")
ax.set_xlabel("Lag (subpulse units)")
ax.set_ylabel("Effective lag weight")
ax.set_title("Conventional gating vs multipulse: lag weighting comparison")
ax.legend()
fig.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Ion-line spectrum fitting / 이온선 스펙트럼 적합 | Lejeune (1979, 1982) and Silen (1981) FORTRAN packages running on NORD-500; iterative comparison of measured ACF with theoretical ACF modified for finite pulse and receiver. / NORD-500에서 동작하는 Lejeune-Silen FORTRAN 패키지의 반복 적합. | GUISDAP (Lehtinen and Huuskonen 1996), pyEIScat, ISR_SIM Python frameworks. / GUISDAP, pyEIScat, ISR_SIM Python 프레임워크. |
| Tristatic 3-D drift / 삼정점 3차원 표류 | PLAN tool (Williams 1982) builds the 3x3 geometry matrix; correlator computes complex cross-ACF in real time. / PLAN(Williams 1982)이 3x3 기하 행렬 구축, 상관기가 실시간 복소 교차 ACF 계산. | EISCAT_3D vector measurements, MIMO digital beamforming, AMISR PFISR plasma drift maps. / EISCAT_3D 벡터 측정, MIMO 디지털 빔포밍, AMISR PFISR 플라스마 드리프트 지도. |
| Multipulse autocorrelation / 다중펄스 자기상관 | Custom microprograms by Ho (1981) and Kofman (1982); Barker plus multipulse for 3-km range resolution. / Ho(1981)와 Kofman(1982)의 맞춤 마이크로프로그램, 3 km 분해능을 위한 Barker+multipulse. | Modern alternating codes, randomised codes, full-profile maximum-likelihood estimation. / 현대 alternating code, randomised code, full-profile 최대우도 추정. |
| Real-time control / 실시간 제어 | EROS OS, TARLAN compiler, CORLAN correlator language on NORD-10. / NORD-10에서 EROS OS, TARLAN 컴파일러, CORLAN 상관기 언어. | EROS-3 (modernised), Linux-based AMISR-S Sondrestrom controllers, EISCAT_3D digital site control. / EROS-3, Linux 기반 AMISR-S 제어기, EISCAT_3D 디지털 사이트 제어. |